# 04c. Decompor sem CPF

Depois do [`04_atribuir.ipynb`](04_atribuir.ipynb). Universo: Censo de
**aplicação** que não entrou em `splink_atribuicao`.

Métrica do modelo: atribuídos / aplicação. Os sem CPF são o complemento.
Este notebook parte esse complemento por melhor `p` no parquet e, quando a
nota era alta o bastante para a escada, por motivo (cota, veto, sem
corroborador, resto). Quem não tem `p ≥ 0,5` separa blocking (14 regras do
02b) vs sem bloco.

Não altera atribuição nem modelo. Níveis de nome/data na faixa baixa:
[`04b_avaliar_nota_baixa.ipynb`](04b_avaliar_nota_baixa.ipynb).


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    MAX_CENSOS_POR_CPF,
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO_ESCADA = 0.75
PISO_PARQUET = 0.25
TETO = MAX_CENSOS_POR_CPF

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print('Piso escada 04:', PISO_ESCADA, '| teto Censos/CPF:', TETO)


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

splink_input → censo_limpo_aplicacao ∪ cpf_limpo_aplicacao
Piso escada 04: 0.75 | teto Censos/CPF: 3


In [2]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_2 AS
SELECT unique_id_censo, unique_id_cpf, match_probability, degrau, regra
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

n_app = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''').fetchone()[0]
n_atrib = con.execute('SELECT COUNT(*) FROM atribuicao_2').fetchone()[0]
n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao_2 a
    WHERE a.unique_id_censo = c.unique_id
  )
''')
n_sem = con.execute('SELECT COUNT(*) FROM censo_sem_cpf').fetchone()[0]

print(f'Aplicação: {n_app:,}')
print(f'Atribuídos: {n_atrib:,} ({100.0 * n_atrib / n_app:.1f}%)')
print(f'Sem CPF: {n_sem:,} ({100.0 * n_sem / n_app:.1f}%)')
print(f'Pares no parquet: {n_pred:,}')
assert n_atrib + n_sem == n_app


Aplicação: 6,079,030
Atribuídos: 4,092,388 (67.3%)
Sem CPF: 1,986,642 (32.7%)
Pares no parquet: 6,598,211


Melhor par no parquet por Censo sem CPF. Fatias de `p` — quem tem nota alta e
mesmo assim ficou de fora não é o mesmo buraco de quem não tem `p ≥ 0,25`.


In [3]:
con.execute('''
CREATE OR REPLACE TABLE melhor_par AS
SELECT
    s.unique_id AS unique_id_censo,
    p.unique_id_cpf,
    p.match_probability
FROM censo_sem_cpf s
LEFT JOIN (
    SELECT unique_id_censo, unique_id_cpf, match_probability
    FROM splink_predictions
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY unique_id_censo
        ORDER BY match_probability DESC, unique_id_cpf
    ) = 1
) p ON p.unique_id_censo = s.unique_id
''')

con.execute('''
CREATE OR REPLACE TABLE melhor_par_faixa AS
SELECT
    unique_id_censo,
    unique_id_cpf,
    match_probability,
    CASE
        WHEN match_probability IS NULL THEN 'sem_par_025'
        WHEN match_probability >= 0.99 THEN 'p_ge_099'
        WHEN match_probability >= 0.75 THEN 'p_075_099'
        WHEN match_probability >= 0.50 THEN 'p_050_075'
        WHEN match_probability >= 0.25 THEN 'p_025_050'
        ELSE 'sem_par_025'
    END AS faixa_p
FROM melhor_par
''')

display(con.execute(f'''
SELECT
    a.faixa_p,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / {n_sem}, 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'p_ge_099'),
    (2, 'p_075_099'),
    (3, 'p_050_075'),
    (4, 'p_025_050'),
    (5, 'sem_par_025')
) a(ord, faixa_p)
LEFT JOIN (
    SELECT faixa_p, COUNT(*) AS n
    FROM melhor_par_faixa
    GROUP BY 1
) g ON g.faixa_p = a.faixa_p
ORDER BY a.ord
''').df())

n_faixa = con.execute('SELECT COUNT(*) FROM melhor_par_faixa').fetchone()[0]
assert n_faixa == n_sem


,faixa_p,n,pct_sem_cpf,pct_aplicacao
0,p_ge_099,290,0.0,0.0
1,p_075_099,203869,10.3,3.4
2,p_050_075,63110,3.2,1.0
3,p_025_050,79005,4.0,1.3
4,sem_par_025,1640368,82.6,27.0


Para quem tem melhor `p ≥ 0,75` (piso da escada do 04): motivo aproximado no
**melhor par**, com as mesmas regras de elegibilidade e corroborador do 04.

- `veto_mae_ou_primeiro` — não passaria em `pares_ok` (primeiro nome ou mãe)
- `cota_cheia` — o CPF do melhor par já tem 3 Censos na atribuição final
- `sem_corroborador` — `p < 0,99` e nenhuma regra de corroborador
- `outro_escada` — elegível e com cota, mas não saiu (1:1 do degrau, lote que
  estouraria a cota no insert, outro Censo levou o slot, etc.)

Abaixo de 0,75: `abaixo_piso_075` ou `faixa_baixa_025_050`. Sem par ≥ 0,25:
fica para a célula de blocking.


In [4]:
con.execute(f'''
CREATE OR REPLACE TABLE top10_primeiro AS
SELECT primeiro_nome_phon
FROM {SPLINK_INPUT_VIEW}
WHERE origem = 'censo'
  AND primeiro_nome_phon IS NOT NULL
  AND primeiro_nome_phon <> ''
GROUP BY 1
ORDER BY COUNT(*) DESC, primeiro_nome_phon
LIMIT 10
''')

con.execute(f'''
CREATE OR REPLACE TABLE top10_primeiro_mae AS
SELECT primeiro_nome_mae_phon
FROM {SPLINK_INPUT_VIEW}
WHERE origem = 'censo'
  AND primeiro_nome_mae_phon IS NOT NULL
  AND primeiro_nome_mae_phon <> ''
GROUP BY 1
ORDER BY COUNT(*) DESC, primeiro_nome_mae_phon
LIMIT 10
''')

n_mae = (
    "least(len(string_split(ca.nome_mae_phon, ' ')), "
    "len(string_split(pb.nome_mae_phon, ' ')))"
)
prefixo_mae = (
    f"{n_mae} >= 2 AND "
    f"list_slice(string_split(ca.nome_mae_phon, ' '), 1, {n_mae}) = "
    f"list_slice(string_split(pb.nome_mae_phon, ' '), 1, {n_mae})"
)
mae_jw = 'jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) >= 0.75'
mae_igual_ou_prefixo = (
    f'ca.nome_mae_phon = pb.nome_mae_phon OR ({prefixo_mae})'
)
mae_contem = (
    f"{n_mae} >= 2 AND "
    f"len(string_split(ca.nome_mae_phon, ' ')) <> len(string_split(pb.nome_mae_phon, ' ')) AND "
    f"len(list_intersect(string_split(ca.nome_mae_phon, ' '), string_split(pb.nome_mae_phon, ' '))) = {n_mae} AND "
    f"NOT ({prefixo_mae})"
)
mae_contem_3 = (
    f"{n_mae} >= 3 AND "
    f"len(string_split(ca.nome_mae_phon, ' ')) <> len(string_split(pb.nome_mae_phon, ' ')) AND "
    f"len(list_intersect(string_split(ca.nome_mae_phon, ' '), string_split(pb.nome_mae_phon, ' '))) = {n_mae} AND "
    f"NOT ({prefixo_mae})"
)
mae_um_nome = (
    f"{n_mae} = 1 AND "
    f"len(string_split(ca.nome_mae_phon, ' ')) <> len(string_split(pb.nome_mae_phon, ' ')) AND "
    f"len(list_intersect(string_split(ca.nome_mae_phon, ' '), string_split(pb.nome_mae_phon, ' '))) = 1"
)
data_nula = 'ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL'
data_nula_ou_diferente = (
    f"({data_nula}) OR ("
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento <> pb.data_nascimento "
    "AND abs(year(CAST(ca.data_nascimento AS DATE)) - "
    "year(CAST(pb.data_nascimento AS DATE))) <= 10)"
)
mae_nao_frequente = (
    "ca.primeiro_nome_mae_phon NOT IN ("
    "SELECT primeiro_nome_mae_phon FROM top10_primeiro_mae)"
)
mae_baixa_nota = (
    f"m.match_probability > 0.05 AND m.match_probability < 0.50 "
    f"AND ({data_nula_ou_diferente}) AND ("
    f"(({mae_jw}) AND ({mae_nao_frequente})) OR ({mae_contem_3})"
    f")"
)
mae_data_nula = (
    f"m.match_probability >= 0.35 AND ({data_nula}) "
    f"AND (({mae_jw}) OR ({mae_contem}))"
)
mae_passa = (
    f"({mae_igual_ou_prefixo}) "
    f"OR (m.match_probability >= 0.50 AND ({mae_jw})) "
    f"OR (m.match_probability >= 0.50 AND ({mae_contem})) "
    f"OR (m.match_probability >= 0.90 AND ({mae_um_nome})) "
    f"OR ({mae_data_nula}) "
    f"OR ({mae_baixa_nota})"
)
n_nome = (
    "least(len(string_split(ca.nome_completo_phon, ' ')), "
    "len(string_split(pb.nome_completo_phon, ' ')))"
)
prefixo_nome = (
    f"{n_nome} >= 2 AND "
    f"list_slice(string_split(ca.nome_completo_phon, ' '), 1, {n_nome}) = "
    f"list_slice(string_split(pb.nome_completo_phon, ' '), 1, {n_nome})"
)
nao_frequente = (
    "ca.primeiro_nome_phon NOT IN (SELECT primeiro_nome_phon FROM top10_primeiro)"
)
sobreposicao = (
    "ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL AND "
    "len(list_intersect("
    "string_split(ca.nome_completo_phon, ' '), "
    "string_split(pb.nome_completo_phon, ' ')"
    ")) * 1.0 / greatest("
    "len(string_split(ca.nome_completo_phon, ' ')), "
    "len(string_split(pb.nome_completo_phon, ' '))"
    ") >= 0.8"
)
primeiro_proporcional = (
    "least(length(ca.primeiro_nome_phon), length(pb.primeiro_nome_phon)) >= 6 "
    "AND damerau_levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) <= 2 "
    "AND damerau_levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) * 6 "
    "<= least(length(ca.primeiro_nome_phon), length(pb.primeiro_nome_phon))"
)
tok_c = "string_split(ca.nome_completo_phon, ' ')"
tok_p = "string_split(pb.nome_completo_phon, ' ')"
n_tok = f"len({tok_c})"
palavras_diferentes = (
    "list_aggregate(list_transform("
    f"list_zip({tok_c}, {tok_p}), "
    "x -> CASE WHEN x[1] <> x[2] THEN 1 ELSE 0 END"
    "), 'sum')"
)
uma_palavra = (
    f"{n_tok} = len({tok_p}) AND {n_tok} >= 2 AND {palavras_diferentes} <= 1 "
    f"AND NOT ({n_tok} = 3 AND {tok_c}[1] = {tok_p}[1] "
    f"AND {tok_c}[3] = {tok_p}[3] AND {tok_c}[2] <> {tok_p}[2])"
)
nome_sem_espaco = (
    "ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL AND ("
    "replace(ca.nome_completo_phon, ' ', '') = replace(pb.nome_completo_phon, ' ', '') "
    "OR damerau_levenshtein("
    "replace(ca.nome_completo_phon, ' ', ''), "
    "replace(pb.nome_completo_phon, ' ', '')) <= 1)"
)
data_proxima = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL AND ("
    "ca.data_nascimento = pb.data_nascimento "
    "OR damerau_levenshtein(CAST(ca.data_nascimento AS VARCHAR), "
    "CAST(pb.data_nascimento AS VARCHAR)) <= 1 "
    "OR ("
    "month(CAST(ca.data_nascimento AS DATE)) = month(CAST(pb.data_nascimento AS DATE)) "
    "AND day(CAST(ca.data_nascimento AS DATE)) = day(CAST(pb.data_nascimento AS DATE)) "
    "AND abs(year(CAST(ca.data_nascimento AS DATE)) - "
    "year(CAST(pb.data_nascimento AS DATE))) = 1))"
)

con.execute(f'''
CREATE OR REPLACE TABLE cpf_n_atrib AS
SELECT unique_id_cpf, COUNT(*) AS n_censo
FROM atribuicao_2
GROUP BY 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_par_motivo AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    m.faixa_p,
    CASE
        WHEN m.faixa_p = 'sem_par_025' THEN 'sem_par_025'
        WHEN m.faixa_p = 'p_025_050' THEN 'faixa_baixa_025_050'
        WHEN m.faixa_p = 'p_050_075' THEN 'abaixo_piso_075'
        WHEN (
                (m.match_probability < 0.95 OR m.match_probability >= 0.99)
                AND (
                    ca.primeiro_nome_phon IS NULL
                    OR pb.primeiro_nome_phon IS NULL
                    OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) > 1
                )
                AND NOT (
                    ({data_proxima}) AND ({nao_frequente})
                    AND (({uma_palavra}) OR ({nome_sem_espaco}))
                )
             )
             OR (
                m.match_probability < 0.95
                AND ca.nome_mae_phon IS NOT NULL
                AND pb.nome_mae_phon IS NOT NULL
                AND NOT ({mae_passa})
                AND NOT (
                    ca.nome_completo_phon IS NOT NULL
                    AND pb.nome_completo_phon IS NOT NULL
                    AND ca.nome_completo_phon = pb.nome_completo_phon
                    AND ca.data_nascimento IS NOT NULL
                    AND pb.data_nascimento IS NOT NULL
                    AND ca.data_nascimento = pb.data_nascimento
                )
             )
            THEN 'veto_mae_ou_primeiro'
        WHEN coalesce(c.n_censo, 0) >= {TETO}
            THEN 'cota_cheia'
        WHEN m.match_probability < 0.99
             AND NOT (
                (ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
                 AND ({mae_passa}))
                OR (ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
                    AND ca.logradouro_norm = pb.logradouro_norm
                    AND ca.cod_municipio IS NOT NULL AND pb.cod_municipio IS NOT NULL
                    AND ca.cod_municipio = pb.cod_municipio)
                OR (ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
                    AND ca.data_nascimento = pb.data_nascimento
                    AND ({prefixo_nome}))
                OR (ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
                    AND ca.data_nascimento = pb.data_nascimento
                    AND ca.ultimo_nome_phon IS NOT NULL AND pb.ultimo_nome_phon IS NOT NULL
                    AND ca.ultimo_nome_phon = pb.ultimo_nome_phon
                    AND ({primeiro_proporcional}))
                OR (ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL
                    AND ca.nome_completo_phon = pb.nome_completo_phon
                    AND ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
                    AND ({nao_frequente}))
                OR (({sobreposicao})
                    AND ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
                    AND ({nao_frequente}))
                OR (({data_proxima}) AND ({nao_frequente})
                    AND (({uma_palavra}) OR ({nome_sem_espaco})))
             )
            THEN 'sem_corroborador'
        ELSE 'outro_escada'
    END AS motivo
FROM melhor_par_faixa m
LEFT JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
LEFT JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
LEFT JOIN cpf_n_atrib c ON c.unique_id_cpf = m.unique_id_cpf
''')

display(con.execute(f'''
SELECT
    a.motivo,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / {n_sem}, 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'veto_mae_ou_primeiro'),
    (2, 'cota_cheia'),
    (3, 'sem_corroborador'),
    (4, 'outro_escada'),
    (5, 'abaixo_piso_075'),
    (6, 'faixa_baixa_025_050'),
    (7, 'sem_par_025')
) a(ord, motivo)
LEFT JOIN (
    SELECT motivo, COUNT(*) AS n
    FROM melhor_par_motivo
    GROUP BY 1
) g ON g.motivo = a.motivo
ORDER BY a.ord
''').df())

display(con.execute('''
SELECT
    faixa_p,
    motivo,
    COUNT(*) AS n
FROM melhor_par_motivo
GROUP BY 1, 2
ORDER BY 1, n DESC
''').df())

n_motivo = con.execute('SELECT COUNT(*) FROM melhor_par_motivo').fetchone()[0]
assert n_motivo == n_sem


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,motivo,n,pct_sem_cpf,pct_aplicacao
0,veto_mae_ou_primeiro,27110,1.4,0.4
1,cota_cheia,20,0.0,0.0
2,sem_corroborador,173797,8.7,2.9
3,outro_escada,3232,0.2,0.1
4,abaixo_piso_075,63110,3.2,1.0
5,faixa_baixa_025_050,79005,4.0,1.3
6,sem_par_025,1640368,82.6,27.0


,faixa_p,motivo,n
0,p_025_050,faixa_baixa_025_050,79005
1,p_050_075,abaixo_piso_075,63110
2,p_075_099,sem_corroborador,173797
3,p_075_099,veto_mae_ou_primeiro,26833
4,p_075_099,outro_escada,3219
5,p_075_099,cota_cheia,20
6,p_ge_099,veto_mae_ou_primeiro,277
7,p_ge_099,outro_escada,13
8,sem_par_025,sem_par_025,1640368


Quem está em `sem_par_025`: não tem linha no parquet (`p ≥ 0,25`). As 14 regras
do [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb), na mesma ordem.
`com_bloco` cruzou algum CPF e ficou com score &lt; 0,25; `sem_bloco` não cruzou.


In [5]:
con.execute(f'''
CREATE OR REPLACE TABLE sem_par_025 AS
SELECT c.*
FROM melhor_par_motivo m
JOIN {SPLINK_INPUT_VIEW} c ON c.unique_id = m.unique_id_censo
WHERE m.motivo = 'sem_par_025'
''')

# Mesmas 14 regras de blocking do 02b, na mesma ordem.
con.execute(f'''
CREATE OR REPLACE TABLE sem_par_com_bloco AS
SELECT unique_id FROM (
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.nome_completo_phon = p.nome_completo_phon
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.primeiro_nome_phon = p.primeiro_nome_phon
     AND c.ultimo_nome_phon = p.ultimo_nome_phon
     AND c.ano_nascimento = p.ano_nascimento
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.primeiro_nome_phon = p.primeiro_nome_phon
     AND c.ultimo_nome_phon = p.ultimo_nome_phon
     AND c.mes_nascimento = p.mes_nascimento
     AND c.dia_nascimento = p.dia_nascimento
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.primeiro_nome_phon = p.primeiro_nome_phon
     AND c.ultimo_nome_phon = p.ultimo_nome_phon
     AND c.mes_nascimento = p.mes_nascimento
     AND c.ano_nascimento = p.ano_nascimento
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.primeiro_nome_phon = p.primeiro_nome_phon
     AND c.data_nascimento = p.data_nascimento
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.ultimo_nome_phon = p.ultimo_nome_phon
     AND c.data_nascimento = p.data_nascimento
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.primeiro_nome_phon = p.primeiro_nome_phon
     AND c.mes_nascimento = p.mes_nascimento
     AND c.dia_nascimento = p.dia_nascimento
     AND c.cod_municipio = p.cod_municipio
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.primeiro_nome_phon = p.primeiro_nome_phon
     AND c.mes_nascimento = p.mes_nascimento
     AND c.ano_nascimento = p.ano_nascimento
     AND c.cod_municipio = p.cod_municipio
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.ultimo_nome_phon = p.ultimo_nome_phon
     AND c.mes_nascimento = p.mes_nascimento
     AND c.dia_nascimento = p.dia_nascimento
     AND c.sexo = p.sexo
     AND c.cep = p.cep
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.ultimo_nome_phon = p.ultimo_nome_phon
     AND c.mes_nascimento = p.mes_nascimento
     AND c.ano_nascimento = p.ano_nascimento
     AND c.sexo = p.sexo
     AND c.cep = p.cep
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.ultimo_nome_phon = p.ultimo_nome_phon
     AND c.ano_nascimento = p.ano_nascimento
     AND c.mes_nascimento = p.mes_nascimento
     AND c.sexo = p.sexo
     AND damerau_levenshtein(c.primeiro_nome_phon, p.primeiro_nome_phon) <= 2
     AND least(len(c.primeiro_nome_phon), len(p.primeiro_nome_phon)) >= 6
     AND damerau_levenshtein(c.primeiro_nome_phon, p.primeiro_nome_phon) * 6
         <= least(len(c.primeiro_nome_phon), len(p.primeiro_nome_phon))
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.primeiro_nome_phon = p.primeiro_nome_phon
     AND c.ultimo_nome_phon = p.ultimo_nome_phon
     AND c.cod_municipio = p.cod_municipio
     AND c.sexo = p.sexo
     AND abs(c.idade - p.idade) <= 1
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.logradouro_norm = p.logradouro_norm
     AND c.cep = p.cep
     AND c.cod_municipio = p.cod_municipio
     AND c.sexo = p.sexo
     AND abs(c.idade - p.idade) <= 1
    UNION
    SELECT DISTINCT c.unique_id
    FROM sem_par_025 c
    INNER JOIN {SPLINK_INPUT_VIEW} p
      ON p.origem = 'cpf'
     AND c.nome_mae_phon = p.nome_mae_phon
     AND c.cod_municipio = p.cod_municipio
     AND c.sexo = p.sexo
     AND abs(c.idade - p.idade) <= 1
) t
''')

con.execute('''
CREATE OR REPLACE TABLE sem_par_motivo AS
SELECT
    s.unique_id AS unique_id_censo,
    CASE
        WHEN b.unique_id IS NOT NULL THEN 'com_bloco_p_lt_025'
        ELSE 'sem_bloco'
    END AS motivo_bloco
FROM sem_par_025 s
LEFT JOIN sem_par_com_bloco b ON b.unique_id = s.unique_id
''')

display(con.execute(f'''
SELECT
    a.motivo_bloco,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / nullif(t.n_sem025, 0), 1) AS pct_sem_par_025,
    ROUND(100.0 * coalesce(g.n, 0) / {n_sem}, 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'com_bloco_p_lt_025'),
    (2, 'sem_bloco')
) a(ord, motivo_bloco)
CROSS JOIN (SELECT COUNT(*) AS n_sem025 FROM sem_par_025) t
LEFT JOIN (
    SELECT motivo_bloco, COUNT(*) AS n
    FROM sem_par_motivo
    GROUP BY 1
) g ON g.motivo_bloco = a.motivo_bloco
ORDER BY a.ord
''').df())

n_sem025 = con.execute('SELECT COUNT(*) FROM sem_par_025').fetchone()[0]
n_bloco = con.execute('SELECT COUNT(*) FROM sem_par_motivo').fetchone()[0]
n_com = con.execute(
    "SELECT COUNT(*) FROM sem_par_motivo WHERE motivo_bloco = 'com_bloco_p_lt_025'"
).fetchone()[0]
assert n_bloco == n_sem025
assert n_com + (n_sem025 - n_com) == n_sem025


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,motivo_bloco,n,pct_sem_par_025,pct_sem_cpf,pct_aplicacao
0,com_bloco_p_lt_025,1415385,86.3,71.2,23.3
1,sem_bloco,224983,13.7,11.3,3.7


Partição única dos sem CPF. `sem_par_025` vira `com_bloco_p_lt_025` ou
`sem_bloco`. A soma tem que fechar `n_sem_cpf`.


In [6]:
con.execute('''
CREATE OR REPLACE TABLE sem_cpf_particao AS
SELECT
    m.unique_id_censo,
    m.faixa_p,
    m.match_probability,
    m.unique_id_cpf,
    CASE
        WHEN m.motivo = 'sem_par_025' THEN b.motivo_bloco
        ELSE m.motivo
    END AS motivo
FROM melhor_par_motivo m
LEFT JOIN sem_par_motivo b ON b.unique_id_censo = m.unique_id_censo
''')

resumo = con.execute(f'''
SELECT
    a.motivo,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / {n_sem}, 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'veto_mae_ou_primeiro'),
    (2, 'cota_cheia'),
    (3, 'sem_corroborador'),
    (4, 'outro_escada'),
    (5, 'abaixo_piso_075'),
    (6, 'faixa_baixa_025_050'),
    (7, 'com_bloco_p_lt_025'),
    (8, 'sem_bloco')
) a(ord, motivo)
LEFT JOIN (
    SELECT motivo, COUNT(*) AS n
    FROM sem_cpf_particao
    GROUP BY 1
) g ON g.motivo = a.motivo
ORDER BY a.ord
''').df()
display(resumo)

n_part = int(resumo['n'].sum())
print(f'Soma partição: {n_part:,} | sem CPF: {n_sem:,}')
assert n_part == n_sem

display(con.execute(f'''
SELECT
    {n_app} AS n_aplicacao,
    {n_atrib} AS n_atribuidos,
    ROUND(100.0 * {n_atrib} / {n_app}, 1) AS pct_modelo,
    {n_sem} AS n_sem_cpf,
    ROUND(100.0 * {n_sem} / {n_app}, 1) AS pct_sem_cpf
''').df())


,motivo,n,pct_sem_cpf,pct_aplicacao
0,veto_mae_ou_primeiro,27110,1.4,0.4
1,cota_cheia,20,0.0,0.0
2,sem_corroborador,173797,8.7,2.9
3,outro_escada,3232,0.2,0.1
4,abaixo_piso_075,63110,3.2,1.0
5,faixa_baixa_025_050,79005,4.0,1.3
6,com_bloco_p_lt_025,1415385,71.2,23.3
7,sem_bloco,224983,11.3,3.7


Soma partição: 1,986,642 | sem CPF: 1,986,642


,n_aplicacao,n_atribuidos,pct_modelo,n_sem_cpf,pct_sem_cpf
0,6079030,4092388,67.3,1986642,32.7


In [7]:
con.close()